# IMDB reviews - Text Analysis

## Understanding data

Load functions

In [ ]:
from sklearn.datasets import load_files
import tarfile
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

Load dataset

In [ ]:
# tar = tarfile.open("data/aclImdb_v1.tar.gz", "r:gz")
# tar.extractall("data/")
# tar.close()

In [ ]:
reviews_train = load_files("data/aclImdb/train/", categories=['pos', 'neg'])
text_train, y_train = reviews_train.data, reviews_train.target
print("type of text_train: {}".format(type(text_train)))
print("length of text_train: {}".format(len(text_train)))
print("text_train[1]:\n{}".format(text_train[1]))

load_files returns a dictionary with data, target_names, target as the keys.

Cleaning html breaks (\<br />)

In [ ]:
text_train = [doc.replace(b"<br />", b" ") for doc in text_train]

Check balance of classes

In [ ]:
print("Samples per class (training): {}".format(np.bincount(y_train)))

Load test data

In [ ]:
reviews_test = load_files("data/aclImdb/test", categories=['pos', 'neg'])
text_test, y_test = reviews_test.data, reviews_test.target
print("Number of documents in test data: {}".format(len(text_test)))
print("Samples per class(test): {}".format(np.bincount(y_test)))
text_test = [doc.replace(b"<br />", b" ") for doc in text_test]

## Representing text data as Bag of Words

### example mock dataset

In [ ]:
bards_words =["The fool doth think he is wise,",
 "but the wise man knows himself to be a fool"]

In [ ]:
vect = CountVectorizer()
vect.fit(bards_words)

this consists of tokenization of the training data and building of the vocabulary

In [ ]:
print("Vocabulary size: {}".format(len(vect.vocabulary_)))

In [ ]:
print("Vocabulary content: {}".format(vect.vocabulary_))

BoW representation

In [ ]:
bag_of_words = vect.transform(bards_words)
print("bag_of_words: {}".format(repr(bag_of_words)))

view the sparse bag_of_words matrix

In [ ]:
print("Dense representation:\n{}".format(bag_of_words.toarray()))

## Bag-of-Words for Movie Reviews

In [ ]:
vect = CountVectorizer().fit(text_train)
X_train = vect.transform(text_train)
print("X_train:\n{}".format(repr(X_train)))

look at the features in detail

In [ ]:
feature_names = vect.get_feature_names_out()
print("Number of features: {}".format(len(feature_names)))
print("First 25 features:\n{}".format(feature_names[:25]))
print("Features 20010 to 20030:\n{}".format(feature_names[20010:20030]))
print("Every 3000th feature:\n{}".format(feature_names[::3000]))

get any 5 reviews with word '007'

In [ ]:
indices = np.where(X_train[:,vect.vocabulary_.get('007', -1)].toarray()>0)[0]
# Print the first 5 reviews with '007'
for i in indices[:5]:
  print(f"Review {i+1}: {text_train[i]}")

## Baseline classifier


In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

In [ ]:
scores = cross_val_score(LogisticRegression(max_iter=2000), X_train, y_train, cv=5)

In [ ]:
print(f"Mean cross-validation accuracy: {np.mean(scores):.2f}")

Hyperparameter tuning

In [ ]:
param = {'C' : [.001, .01, .1, 1, 10]}
grid = GridSearchCV(LogisticRegression(max_iter=500), param, cv=5)
grid.fit(X_train, y_train)

In [ ]:
print(f"Best cross-validation score: {grid.best_score_:.2f}")
print(f"Best parameters: {grid.best_params_}")

Assessing the performance on the test set

In [ ]:
X_test = vect.transform(text_test)
print(f"Test score: {grid.score(X_test, y_test):.2f}")

## Improve word extraction

set min documents a word should appear in to be included in vocabulary

In [ ]:
vect = CountVectorizer(min_df=5).fit(text_train)
X_train = vect.transform(text_train)
print("X_train with min_df:{}".format(repr(X_train)))

In [ ]:
feature_names = vect.get_feature_names_out()
print("Number of features: {}".format(len(feature_names)))
print("First 50 features:\n{}".format(feature_names[:50]))
print("Features 20010 to 20030:\n{}".format(feature_names[20010:20030]))
print("Every 700th feature:\n{}".format(feature_names[::700]))

checking model performance

In [ ]:
grid = GridSearchCV(LogisticRegression(max_iter=500), param, cv=5)
grid.fit(X_train, y_train)

In [ ]:
print(f"Best cross-validation score: {grid.best_score_:.2f}")
print(f"Best parameters: {grid.best_params_}")

## Stop Words

In [ ]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

In [ ]:
print(f"No. of stop words: {len(ENGLISH_STOP_WORDS)}")
print(f"Every 10th stopword: {list(ENGLISH_STOP_WORDS)[::10]}")

In [ ]:
vect = CountVectorizer(min_df=5, stop_words='english').fit(text_train)
X_train = vect.transform(text_train)
print(f"X_train with stop words: {repr(X_train)}")

In [ ]:
grid = GridSearchCV(LogisticRegression(max_iter=400), param, cv=5)
grid.fit(X_train, y_train)

In [ ]:
print(f"Best cross-validation score: {grid.best_score_:.2f}")

## Experimenting with max_df

baseline with just min_df=5

X_train with max_df: <Compressed Sparse Row sparse matrix of dtype 'int64' with 3354014 stored elements and shape (25000, 27271)>

In [ ]:
max_df = 1000
vect = CountVectorizer(min_df=5, max_df=max_df).fit(text_train)
X_train = vect.transform(text_train)
print(f"X_train with max_df {max_df}: {repr(X_train)}")

X_train with max_df 20000: <Compressed Sparse Row sparse matrix of dtype 'int64' with 3148142 stored elements and shape (25000, 27262)>

In [ ]:
grid = GridSearchCV(LogisticRegression(), param, cv=5)
grid.fit(X_train, y_train)

In [ ]:
print(f"Best cross-validation score: {grid.best_score_:.2f}")

The model is giving an accuracy score of 0.85 which is not significantly lower than the earlier model's score of 0.89, and also the count of features has gone down from 27271 to 26749.

In [ ]:
27271-26749

## Rescaling the data with tf-idf

creating pipeline


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import make_pipeline

In [ ]:
pipe = make_pipeline(TfidfVectorizer(min_df=5, norm=None),
                     LogisticRegression(max_iter=200))
param = {'logisticregression__C': [.001, .01, .1, 1, 10]}
grid = GridSearchCV(pipe, param, cv=5)
grid.fit(text_train, y_train)

In [ ]:
print(f"Best cross-validation score: {grid.best_score_:.2f}")

In [ ]:
vectorizer = grid.best_estimator_.named_steps["tfidfvectorizer"]
X_train = vectorizer.transform(text_train)
max_value = X_train.max(axis=0).toarray().ravel()
sorted_by_tfidf = max_value.argsort()
feature_names = np.array(vectorizer.get_feature_names_out())
print(f"Features with lowest tfidf:\n{feature_names[sorted_by_tfidf[:20]]}")
print(f"Features with highest tfidf:\n{feature_names[sorted_by_tfidf[-20:]]}")

words that have low idf

In [ ]:
sorted_by_idf = np.argsort(vectorizer.idf_)
print(f"Features with lowest idf:\n{feature_names[sorted_by_idf[:100]]}")

## Investigating Model Coefficients

In [ ]:
logitreg = grid.best_estimator_.named_steps['logisticregression']
coeff = logitreg.coef_[0]
sorted_coeff_index = np.argsort(coeff)
smallest_coeff = sorted_coeff_index[:25]
largest_coeff = sorted_coeff_index[-25:]
coeff_to_plot = np.concatenate((smallest_coeff, largest_coeff))
coeff_to_plot_names = np.concatenate((feature_names[smallest_coeff], feature_names[largest_coeff]))
coeff_to_plot_values = np.concatenate((coeff[smallest_coeff], coeff[largest_coeff]))

In [ ]:
plt.figure(figsize=(12,6))
plt.bar(range(len(coeff_to_plot)), coeff_to_plot_values, align='center')
plt.xticks(range(len(coeff_to_plot)), coeff_to_plot_names, rotation=75)
plt.title('25 Largest and 25 Smallest Coefficients of Logistic Regression Model')
plt.xlabel('Features')
plt.ylabel('Coefficients')
plt.tight_layout()
plt.show()

## using more than a word (n-grams)

In [ ]:
import tempfile

with tempfile.TemporaryDirectory() as temp_dir:
    pipe = make_pipeline(
        TfidfVectorizer(min_df=5),
        LogisticRegression(solver='saga'),
        memory=temp_dir)
    param = {'logisticregression__C': [0.1, 1, 10, 100],
         'tfidfvectorizer__ngram_range': [(1,1), (1,2), (1,3)]}
    grid = GridSearchCV(pipe, param, cv=5, verbose=True)
    grid.fit(text_train, y_train)

In [ ]:
print(f"Best cross-validation score: {grid.best_score_:.2f}")
print(f"Best parameters:\n{grid.best_params_}")

In [ ]:
result = pd.DataFrame(grid.cv_results_)
result.head()

In [ ]:
scores = grid.cv_results_['mean_test_score'].reshape(-1,3)
scores

In [ ]:
import seaborn as sns
scores = grid.cv_results_['mean_test_score'].reshape(-1,3).T
fig, ax = plt.subplots(figsize=(4,3))
ax = sns.heatmap(scores,
                      xticklabels=param['logisticregression__C'],
                      yticklabels=param['tfidfvectorizer__ngram_range'],
                      fmt='.4f', annot=True, linewidth=.5)
ax.set_xlabel('C')
ax.set_ylabel('ngram_range')
plt.show()

In [ ]:
vect = grid.best_estimator_.named_steps['tfidfvectorizer']
feature_names = np.array(vect.get_feature_names_out())
coef = grid.best_estimator_.named_steps['logisticregression'].coef_[0]
coef_sorted_idx = np.argsort(coef)
n_features = 40
coef_sorted_idx_top = coef_sorted_idx[-n_features:]
coef_sorted_idx_low = coef_sorted_idx[:n_features]

In [ ]:
fig = plt.figure(figsize=(15,4))

plt.bar(range(-n_features,0), coef[coef_sorted_idx_low], color='r', edgecolor='k')
plt.bar(range(0, n_features), coef[coef_sorted_idx_top], color='b', edgecolor='k')
plt.xticks(range(-n_features, n_features), feature_names[np.hstack([coef_sorted_idx_low, coef_sorted_idx_top])], rotation=60, ha='right', size='small')
plt.xlabel('Feature')
plt.ylabel('Coefficient Magnitude')

## Advanced Tokenisation

In [ ]:
import spacy
import nltk

In [ ]:
en_nlp = spacy.load('en_core_web_sm')
stemmer = nltk.stem.PorterStemmer()

In [ ]:
def compare_normalization(doc):
  doc_spacy = en_nlp(doc)
  print("Lemmatization:")
  print([token.lemma_ for token in doc_spacy])

  print('Stemming:')
  print([stemmer.stem(token.norm_.lower()) for token in doc_spacy])


In [ ]:
compare_normalization(u"Our meeting today was worse than yesterday, "
 "I'm scared of meeting the clients tomorrow.")

## lemmatization with scikit-learn

In [ ]:
import re
from tqdm import tqdm

regexp = re.compile('(?u)\\b\\w\\w+\\b')
en_nlp = spacy.load('en_core_web_sm', disable=['parser', 'ner'])

def fast_lemmatizer(documents, batch_size=2000, show_progress=True):
  documents = [doc.decode("utf-8") if isinstance(doc, bytes) else doc for doc in documents]
  tokenized_docs = [" ".join(regexp.findall(doc)) for doc in documents]

  pipe = en_nlp.pipe(tokenized_docs, batch_size=batch_size)
  if show_progress:
    pipe = tqdm(pipe, total=len(documents), desc='Lemmatizing')

  lemmatized_docs = [
      " ".join(token.lemma_ for token in doc if not token.is_punct and not token.is_space)
      for doc in pipe
  ]
  return lemmatized_docs

vectorizer = CountVectorizer(min_df=5)

transform text_train using CountVectorizer with lemmatization

In [ ]:
lemmatized_doc = fast_lemmatizer(text_train)

In [ ]:
X_train_lemma = vectorizer.fit_transform(lemmatized_doc)
print(f"X_train_lemma.shape: {X_train_lemma.shape}")

In [ ]:
X_train = vectorizer.fit_transform(text_train)
print(f"X_train.shape: {X_train.shape}")

## Testing on 1% of the data

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit

param_grid = {'C' : [.001, .01, .1, 1, 10]}
cv = StratifiedShuffleSplit(n_splits=5, test_size=0.99, random_state=0)

grid = GridSearchCV(LogisticRegression(), param_grid, cv=cv)
grid.fit(X_train_lemma, y_train)
print(f"Best cross-validation score\nLemmatization: {grid.best_score_:.2f}")

grid.fit(X_train, y_train)
print(f"Best cross-validation score\nStandard CountVectorizer: {grid.best_score_:.2f}")

# Topic Modeling and Document Clustering

## Latent Dirichlet Allocation (LDA)

remove words that appear in at
 least 20 percent of the documents, and we’ll limit the bag-of-words model to the
 10,000 words that are most common after removing the top 20 percent

In [ ]:
vect = CountVectorizer(max_features=10000, max_df=.2)
X = vect.fit_transform(text_train)

In [ ]:
X.shape

In [ ]:
from sklearn.decomposition import LatentDirichletAllocation
lda = LatentDirichletAllocation(n_components=10, learning_method="batch", max_iter=25, random_state=0)
document_topics = lda.fit_transform(X)

In [ ]:
lda.components_.shape

to understand topics, look at the most important words for each of the topics

In [ ]:
sorting = np.argsort(lda.components_, axis=1)[:,::-1]
feature_names = np.array(vect.get_feature_names_out())

sorting

In [ ]:
for i in range(len(lda.components_)):
  print(f"Topic {i}")
  print(feature_names[sorting[i, :10]])

model with 100 topics

In [ ]:
lda100 = LatentDirichletAllocation(n_components=100, learning_method="batch", max_iter=25, random_state=0)
document_topics100 = lda100.fit_transform(X)

In [ ]:
sorting = np.argsort(lda100.components_, axis=1)[:, ::-1]
for i in range(10):
  print(f"Topic {i}")
  print(feature_names[sorting[i, :20]])

Topic 6 - war movies

In [ ]:
document_topics100.shape

In [ ]:
war = np.argsort(document_topics100[:,6])[::-1]
for i in war[:10]:
  #first two sentences
  print( b".".join(text_train[i].split(b".")[:2]) + b".\n")

inspect the topics is to see how much weight each topic gets over
all, by summing the document_topics over all reviews.

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(10,10))
topic_names = [
    f"{i:>2} " + " ".join(words)
    for i, words in enumerate(feature_names[sorting[:,:2]])
]

for col in [0,1]:
  start = col*50
  end = (col+1)*50
  ax[col].barh(np.arange(50), np.sum(document_topics100, axis=0)[start:end])
  ax[col].set_yticks(np.arange(50))
  ax[col].set_yticklabels(topic_names[start:end], ha='left', va='top')
  ax[col].invert_yaxis()
  ax[col].set_xlim(0,1500)
  yax = ax[col].get_yaxis()
  yax.set_tick_params(pad=120)
plt.tight_layout()
plt.show()